# Criação da camada Bronze — Azure Databricks

Ingestão **batch** das 7 tabelas de `br_inep_avaliacao_alfabetizacao` (Base dos Dados, no BigQuery) para tabelas **Delta** na camada Bronze, usando **PySpark**.

A Bronze guarda o dado **cru**, sem transformação, e **só acrescenta** (`append`): cada execução gera um `id_ingestao` novo, então o histórico das rodadas anteriores é preservado.

**Pré-requisitos (já criados no Azure/Google):**
- Secret scope `google-bigquery-scope`, ligado ao Key Vault, com o segredo `google-credentials-json` (JSON da conta de serviço do Google).
- Conta de serviço do Google com `BigQuery Data Viewer` e `BigQuery Job User` no seu projeto. Como a origem fica em **outro projeto** (`basedosdados`), o conector de leitura do Spark também costuma exigir `BigQuery Read Session User` no seu projeto (confira na documentação do conector).

**Destino:** tabelas Delta em `tech_challenge_fase2.bronze` (catálogo fixo, criado na primeira execução; dá para trocar no parâmetro `CATALOGO`).

**Metadados adicionados a cada tabela:** `id_ingestao`, `data_ingestao`, `tabela_origem`.

## Configuração do ambiente

In [0]:
import base64
import json
import time
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (LongType, StringType, StructField, StructType,
                               TimestampType)

# Parâmetros do notebook (em um Job, viram parâmetros da execução).
dbutils.widgets.text("PROJECT_ID", "fiap-512121", "Projeto GCP (cobrança das consultas)")
dbutils.widgets.text("SOURCE_DATASET", "basedosdados.br_inep_avaliacao_alfabetizacao", "Dataset de origem")
dbutils.widgets.text("CATALOGO", "tech_challenge_fase2", "Catálogo (criado se não existir; vazio = o atual)")
dbutils.widgets.text("BRONZE_SCHEMA", "bronze", "Schema da Bronze")
dbutils.widgets.text("SECRET_SCOPE", "google-bigquery-scope", "Secret scope")
dbutils.widgets.text("SECRET_KEY", "google-credentials-json", "Nome do segredo")


def variavel(nome, padrao=None):
    """Lê um parâmetro do notebook. Valor vazio conta como 'não definido'."""
    valor = dbutils.widgets.get(nome) or padrao
    if valor is None:
        raise RuntimeError(f"Parâmetro {nome} não definido. Preencha o widget no topo do notebook.")
    return valor


PROJECT_ID = variavel("PROJECT_ID")
SOURCE_DATASET = variavel("SOURCE_DATASET")
CATALOGO = variavel("CATALOGO", spark.catalog.currentCatalog())
BRONZE_SCHEMA = variavel("BRONZE_SCHEMA")
SECRET_SCOPE = variavel("SECRET_SCOPE")
SECRET_KEY = variavel("SECRET_KEY")

# Tabelas pequenas primeiro e a `alunos` (a maior) por último: se houver erro de
# permissão ou de conector, ele aparece logo, sem esperar a tabela grande.
TABELAS = ["uf", "meta_alfabetizacao_brasil", "meta_alfabetizacao_uf",
           "meta_alfabetizacao_municipio", "municipio", "dicionario", "alunos"]

print(f"Projeto de cobrança: {PROJECT_ID}")
print(f"Origem: {SOURCE_DATASET}")
print(f"Destino: {CATALOGO}.{BRONZE_SCHEMA}")

Projeto de cobrança: fiap-512121
Origem: basedosdados.br_inep_avaliacao_alfabetizacao
Destino: tech_challenge_fase2.bronze


## Autenticação com o Google

A chave JSON da conta de serviço vem do **Key Vault**, pelo secret scope. Ela nunca é escrita no código nem impressa.

In [0]:
chave_json = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)

# Conferência sem expor a chave: só o e-mail da conta de serviço.
print("Conta de serviço:", json.loads(chave_json)["client_email"])

# O conector de BigQuery do Spark recebe a credencial em Base64.
CREDENCIAIS_B64 = base64.b64encode(chave_json.encode("utf-8")).decode("utf-8")

Conta de serviço: leitura-bq-azure@fiap-512121.iam.gserviceaccount.com


## Funções de ingestão

In [0]:
def carregar_tabela_bq(nome_tabela, dataset=SOURCE_DATASET):
    """Lê uma tabela do BigQuery e devolve um DataFrame Spark (leitura preguiçosa)."""
    return (
        spark.read.format("bigquery")
        .option("table", f"{dataset}.{nome_tabela}")   # projeto.dataset.tabela
        .option("parentProject", PROJECT_ID)           # projeto que recebe a cobrança
        .option("credentials", CREDENCIAIS_B64)
        .load()
    )


def adicionar_metadados_ingestao(df, nome_tabela, id_ingestao, data_ingestao):
    """Adiciona as colunas de metadados de ingestão ao DataFrame."""
    return (
        df.withColumn("id_ingestao", F.lit(id_ingestao))
        .withColumn("data_ingestao", F.lit(data_ingestao))
        .withColumn("tabela_origem", F.lit(nome_tabela))
    )


def nome_tabela_bronze(nome_tabela):
    return f"`{CATALOGO}`.`{BRONZE_SCHEMA}`.`{nome_tabela}`"


def gravar_bronze(df, nome_tabela):
    """Acrescenta o DataFrame à tabela Delta da Bronze, sem sobrescrever rodadas anteriores."""
    destino = nome_tabela_bronze(nome_tabela)
    df.write.format("delta").mode("append").saveAsTable(destino)
    return destino

## Registro das execuções

Cada tabela de cada rodada deixa uma linha em `_ingestion_log` (linhas gravadas, duração, status e erro). A tabela **acumula** as execuções.

In [0]:
LOG_TABELA = nome_tabela_bronze("_ingestion_log")

SCHEMA_LOG = StructType([
    StructField("id_ingestao", StringType()),
    StructField("data_ingestao", TimestampType()),
    StructField("tabela_origem", StringType()),
    StructField("tabela_destino", StringType()),
    StructField("linhas_gravadas", LongType()),
    StructField("duracao_segundos", LongType()),
    StructField("status", StringType()),
    StructField("erro", StringType()),
])


def registrar_execucao(id_ingestao, data_ingestao, nome_tabela, destino, linhas, duracao, status, erro):
    linha = [(id_ingestao, data_ingestao, nome_tabela, destino, linhas, int(duracao), status, erro)]
    spark.createDataFrame(linha, schema=SCHEMA_LOG).write.format("delta").mode("append").saveAsTable(LOG_TABELA)

## Execução da rodada

In [0]:
# Catálogo fixo, para a Bronze não depender do catálogo gerado automaticamente.
try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS `{CATALOGO}`")
except Exception as e:
    raise RuntimeError(
        f"Não foi possível criar o catálogo '{CATALOGO}'. Informe um catálogo que já exista "
        f"no parâmetro CATALOGO ou conceda a permissão CREATE CATALOG. Detalhe: {e}"
    )
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOGO}`.`{BRONZE_SCHEMA}`")

# Um id e um horário por RODADA: todas as tabelas desta execução compartilham os dois.
INGESTION_ID = str(uuid.uuid4())
INGESTION_TS = datetime.now(timezone.utc)
print(f"Rodada {INGESTION_ID} iniciada em {INGESTION_TS.isoformat()}")

for nome in TABELAS:
    inicio = time.time()
    status, erro, linhas, destino = "OK", None, None, None
    try:
        print(f"[{nome}] lendo do BigQuery...")
        df = carregar_tabela_bq(nome)
        df = adicionar_metadados_ingestao(df, nome, INGESTION_ID, INGESTION_TS)

        print(f"[{nome}] gravando na Bronze...")
        destino = gravar_bronze(df, nome)

        linhas = (spark.table(destino)
                  .where(F.col("id_ingestao") == INGESTION_ID)
                  .count())
        print(f"[{nome}] ok: {linhas} linhas em {time.time() - inicio:.0f}s")
    except Exception as e:
        status, erro = "ERRO", str(e)[:1000]
        print(f"[{nome}] ERRO: {erro}")
        raise
    finally:
        registrar_execucao(INGESTION_ID, INGESTION_TS, nome, destino, linhas,
                           time.time() - inicio, status, erro)

Rodada e0a73a22-aa1f-4a00-add9-cc1b7347602c iniciada em 2026-10-20T19:45:18.293551+00:00
[uf] lendo do BigQuery...
[uf] gravando na Bronze...
[uf] ok: 145 linhas em 3s
[meta_alfabetizacao_brasil] lendo do BigQuery...
[meta_alfabetizacao_brasil] gravando na Bronze...
[meta_alfabetizacao_brasil] ok: 3 linhas em 3s
[meta_alfabetizacao_uf] lendo do BigQuery...
[meta_alfabetizacao_uf] gravando na Bronze...
[meta_alfabetizacao_uf] ok: 81 linhas em 3s
[meta_alfabetizacao_municipio] lendo do BigQuery...
[meta_alfabetizacao_municipio] gravando na Bronze...
[meta_alfabetizacao_municipio] ok: 10704 linhas em 3s
[municipio] lendo do BigQuery...
[municipio] gravando na Bronze...
[municipio] ok: 23995 linhas em 11s
[dicionario] lendo do BigQuery...
[dicionario] gravando na Bronze...
[dicionario] ok: 27 linhas em 3s
[alunos] lendo do BigQuery...
[alunos] gravando na Bronze...
[alunos] ok: 3867999 linhas em 7s


## Verificação

Para provar o histórico (Tarefa 4 do desafio), **rode o notebook uma segunda vez**: cada tabela deve mostrar **dois** `id_ingestao` distintos, e a primeira rodada continua lá.

In [0]:
print("Log desta rodada:")
display(spark.table(LOG_TABELA).where(F.col("id_ingestao") == INGESTION_ID))

print("Linhas por rodada em cada tabela da Bronze:")
for nome in TABELAS:
    print(nome)
    display(spark.table(nome_tabela_bronze(nome))
            .groupBy("id_ingestao", "data_ingestao").count()
            .orderBy("data_ingestao"))

Log desta rodada:


id_ingestao,data_ingestao,tabela_origem,tabela_destino,linhas_gravadas,duracao_segundos,status,erro
e0a73a22-aa1f-4a00-add9-cc1b7347602c,2026-10-20T19:45:18.293Z,meta_alfabetizacao_municipio,`tech_challenge_fase2`.`bronze`.`meta_alfabetizacao_municipio`,10704,2,OK,null
e0a73a22-aa1f-4a00-add9-cc1b7347602c,2026-10-20T19:45:18.293Z,meta_alfabetizacao_brasil,`tech_challenge_fase2`.`bronze`.`meta_alfabetizacao_brasil`,3,2,OK,null
e0a73a22-aa1f-4a00-add9-cc1b7347602c,2026-10-20T19:45:18.293Z,meta_alfabetizacao_uf,`tech_challenge_fase2`.`bronze`.`meta_alfabetizacao_uf`,81,2,OK,null
e0a73a22-aa1f-4a00-add9-cc1b7347602c,2026-10-20T19:45:18.293Z,dicionario,`tech_challenge_fase2`.`bronze`.`dicionario`,27,3,OK,null
e0a73a22-aa1f-4a00-add9-cc1b7347602c,2026-10-20T19:45:18.293Z,municipio,`tech_challenge_fase2`.`bronze`.`municipio`,23995,11,OK,null
e0a73a22-aa1f-4a00-add9-cc1b7347602c,2026-10-20T19:45:18.293Z,alunos,`tech_challenge_fase2`.`bronze`.`alunos`,3867999,7,OK,null
e0a73a22-aa1f-4a00-add9-cc1b7347602c,2026-10-20T19:45:18.293Z,uf,`tech_challenge_fase2`.`bronze`.`uf`,145,3,OK,null


Linhas por rodada em cada tabela da Bronze:
uf


id_ingestao,data_ingestao,count
ecace320-575c-4466-8d6d-dc1c177f55c6,2026-10-20T19:42:18.471Z,145
e0a73a22-aa1f-4a00-add9-cc1b7347602c,2026-10-20T19:45:18.293Z,145


meta_alfabetizacao_brasil


...

id_ingestao,data_ingestao,count
ecace320-575c-4466-8d6d-dc1c177f55c6,2026-10-20T19:42:18.471Z,23995
e0a73a22-aa1f-4a00-add9-cc1b7347602c,2026-10-20T19:45:18.293Z,23995


dicionario


id_ingestao,data_ingestao,count
ecace320-575c-4466-8d6d-dc1c177f55c6,2026-10-20T19:42:18.471Z,27
e0a73a22-aa1f-4a00-add9-cc1b7347602c,2026-10-20T19:45:18.293Z,27


alunos


id_ingestao,data_ingestao,count
ecace320-575c-4466-8d6d-dc1c177f55c6,2026-10-20T19:42:18.471Z,3867999
e0a73a22-aa1f-4a00-add9-cc1b7347602c,2026-10-20T19:45:18.293Z,3867999
